# Zero-Shot Object Detection for Rijksmuseum ERP Images and Pascal VOC Export

This notebook demonstrates how to use the Grounding DINO model for zero-shot object detection on a collection of images. It then processes these detections to generate bounding box annotations in the Pascal VOC XML format, which is widely used for training object detection models and compatible with annotation tools like RectLabel.

## Setup: Libraries, Data Loading, and Model Initialization


In [1]:
import os
import torch
import pandas as pd
from PIL import Image, ImageDraw
from io import BytesIO
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from google.colab import drive

In [2]:
import pandas as pd

#Mount Drive & data
drive.mount('/content/drive')
csv_path = '/content/drive/MyDrive/ERP Rijksmuseum /DINO/Image Annotation_Final.xlsx'
image_folder = '/content/drive/MyDrive/ERP Rijksmuseum /DINO/actual interiors (manual)'
df = pd.read_excel(csv_path)

# Clean the DataFrame Picture_IDs
df['Picture ID'] = df['Picture ID'].astype(str).str.strip()

Mounted at /content/drive


In [3]:
#Load DINO
model_id = "IDEA-Research/grounding-dino-tiny"
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/689M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

## Prepare Filename Mapping

This code creates a dictionary (`filename_map`) that maps cleaned image IDs (from your annotation DataFrame) to the actual full file paths of the images stored in your Google Drive. This mapping is crucial for easily locating and processing each image during object detection.

In [4]:
# Create a mapping of canonical filename -> actual path in Drive
image_files_in_folder = os.listdir(image_folder)
filename_map = {}

for filename in image_files_in_folder:
    # Remove "Copy of " to match CSV IDs
    clean_name = filename.replace('Copy of ', '') if filename.startswith('Copy of ') else filename
    filename_map[clean_name] = os.path.join(image_folder, filename)

print(f"Mapped {len(filename_map)} images.")

Mapped 112 images.


## Define Object Detection Function

In [5]:
#detection function
def detect_objects(image_path, text_query, threshold=0.3):
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, text=[[text_query]], return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        results = processor.post_process_grounded_object_detection(
            outputs,
            threshold=threshold,
            text_threshold=threshold,
            target_sizes=[(image.height, image.width)]
        )
        return results[0]['boxes'].tolist()
    except Exception as e:
        return []

## Object detection in corpus

This section iterates through each row of the corpus, applying the `detect_objects` function to find specified objects within the corresponding images. The bounding box coordinates are outputted in a new 'spatial information' column

In [6]:
df['spatial information'] = [[] for _ in range(len(df))]

for index, row in df.iterrows():
    pic_id = row['Picture ID']
    obj_name = row['Object']

    img_path = filename_map.get(pic_id)

    if img_path and pd.notna(obj_name):
        boxes = detect_objects(img_path, str(obj_name))
        df.at[index, 'spatial information'] = boxes

print("Object detection applied to all images in the DataFrame.")

Object detection applied to all images in the DataFrame.


## Export Annotations for RectLabel (Pascal VOC XML Format)

To allow for manual corrections, the 'Spatial Information' for each item in the corpus needs to be exported to fit in RectLabel(Pascal VOC XML format). This code outputs the annotations for each image in an XML file containing the image dimensions and all identified objects, as well as placeholder bounding boxes for annotated objects that couldn't be identified by Groundign DINO.

In [7]:
import os
import shutil
import xml.etree.ElementTree as ET
from xml.dom import minidom
from PIL import Image
import pandas as pd

# Use the full DataFrame for export, ensure imported_annotations_df is aligned
# `df` now contains all detections for the entire dataset
imported_annotations_df = df.copy()

# Define output directories
# This will place the RectLabel_Export folder in your Google Drive
export_base_dir = '/content/drive/MyDrive/RectLabel_Export'
export_image_dir = os.path.join(export_base_dir, 'images')
export_annotation_dir = os.path.join(export_base_dir, 'annotations')

# Create output directories if they don't exist
os.makedirs(export_image_dir, exist_ok=True)
os.makedirs(export_annotation_dir, exist_ok=True)

print(f"Exporting RectLabel compatible files to: {export_base_dir}")
print(f"Images will be copied to: {export_image_dir}")
print(f"Annotations (Pascal VOC XML) will be saved to: {export_annotation_dir}\n")

# Ensure imported_annotations_df and filename_map are available
if imported_annotations_df.empty:
    print("Warning: `imported_annotations_df` is empty. No annotations to export.")
elif 'filename_map' not in globals():
    print("Error: `filename_map` not found. Please run previous cells to define it.")
else:
    # Get unique image IDs from your imported annotations that have actual boxes
    # We don't filter by 'spatial information' here, as we want to export all objects
    unique_image_ids_to_export = imported_annotations_df['Picture ID'].dropna().unique().tolist()

    if not unique_image_ids_to_export:
        print("No images with valid annotations found in `imported_annotations_df` to export.")
    else:
        for image_id in unique_image_ids_to_export:
            img_path = filename_map.get(image_id)
            if not img_path or not os.path.exists(img_path):
                print(f"Warning: Image '{image_id}' not found in filename_map or path invalid. Skipping export for this image.")
                continue

            # 1. Copy Image to export folder
            destination_image_path = os.path.join(export_image_dir, image_id)
            shutil.copy(img_path, destination_image_path)

            # 2. Prepare for XML annotation
            # Filter detections for the current image
            image_specific_annotations = imported_annotations_df[imported_annotations_df['Picture ID'] == image_id]

            # Get image dimensions (needed for XML)
            with Image.open(img_path) as img_pil:
                width, height = img_pil.size

            # Create XML root element
            annotation = ET.Element('annotation')

            # Add common elements
            ET.SubElement(annotation, 'folder').text = 'images' # Or your project name
            ET.SubElement(annotation, 'filename').text = image_id
            # Use a relative path for the XML, as tools often expect images in the 'images' subfolder
            ET.SubElement(annotation, 'path').text = os.path.join('images', image_id)

            source = ET.SubElement(annotation, 'source')
            ET.SubElement(source, 'database').text = 'Unknown'

            size = ET.SubElement(annotation, 'size')
            ET.SubElement(size, 'width').text = str(width)
            ET.SubElement(size, 'height').text = str(height)
            ET.SubElement(size, 'depth').text = '3' # Assuming RGB images

            ET.SubElement(annotation, 'segmented').text = '0'

            # Add objects (bounding boxes)
            for _, row in image_specific_annotations.iterrows():
                obj_name = str(row['Object'])
                # Safely access the bounding box column, assuming it's a list of lists
                detected_boxes = row.get('spatial information', [])

                if not isinstance(detected_boxes, list):
                    # Attempt to parse if it's a string representation of a list
                    try:
                        detected_boxes = eval(detected_boxes) if isinstance(detected_boxes, str) else []
                    except (SyntaxError, TypeError):
                        detected_boxes = []

                # Determine the number of boxes to process based on 'Count'
                num_expected_boxes = 0
                if 'Count' in row and pd.notna(row['Count']):
                    try:
                        int_count = int(float(row['Count']))
                        if int_count >= 0: # Only apply positive counts
                            num_expected_boxes = int_count
                        else:
                            print(f"  Warning: Negative Count '{row['Count']}' for {obj_name} in {image_id}. Processing all found detections.")
                    except ValueError:
                        print(f"  Warning: Non-numeric Count '{row['Count']}' for {obj_name} in {image_id}. Processing all found detections.")

                # If no specific count, or if object name is present, assume at least one instance
                if num_expected_boxes == 0 and pd.notna(obj_name):
                    num_expected_boxes = max(1, len(detected_boxes)) # At least one if object exists, or use actual detections

                # Prepare boxes for export: use detected boxes, add placeholders if needed
                boxes_to_export = detected_boxes[:num_expected_boxes] # Take up to the expected count

                # Add placeholder boxes if detected count is less than expected count
                while len(boxes_to_export) < num_expected_boxes:
                    boxes_to_export.append([0, 0, 1, 1]) # Placeholder bounding box

                for box in boxes_to_export:
                    if len(box) == 4: # Ensure box has 4 coordinates
                        obj = ET.SubElement(annotation, 'object')
                        ET.SubElement(obj, 'name').text = obj_name
                        ET.SubElement(obj, 'pose').text = 'Unspecified'
                        ET.SubElement(obj, 'truncated').text = '0'
                        ET.SubElement(obj, 'difficult').text = '0'

                        bndbox = ET.SubElement(obj, 'bndbox')
                        ET.SubElement(bndbox, 'xmin').text = str(int(box[0]))
                        ET.SubElement(bndbox, 'ymin').text = str(int(box[1]))
                        ET.SubElement(bndbox, 'xmax').text = str(int(box[2]))
                        ET.SubElement(bndbox, 'ymax').text = str(int(box[3]))
                    else:
                        print(f"  Warning: Invalid bounding box format for '{obj_name}' in '{image_id}': {box}. Skipping this box.")

            # Save XML to file
            xml_filename = os.path.splitext(image_id)[0] + '.xml'
            xml_filepath = os.path.join(export_annotation_dir, xml_filename)

            # Pretty print XML
            rough_string = ET.tostring(annotation, 'utf-8')
            reparsed_xml = minidom.parseString(rough_string)
            with open(xml_filepath, 'w', encoding='utf-8') as f:
                f.write(reparsed_xml.toprettyxml(indent="    "))

            print(f"  Exported annotations for {image_id} to {xml_filepath}")

        print("\nRectLabel export process complete! You can find the 'RectLabel_Export' folder in your Google Drive.")

Exporting RectLabel compatible files to: /content/drive/MyDrive/RectLabel_Export
Images will be copied to: /content/drive/MyDrive/RectLabel_Export/images
Annotations (Pascal VOC XML) will be saved to: /content/drive/MyDrive/RectLabel_Export/annotations

  Exported annotations for SK-A-23.jpg to /content/drive/MyDrive/RectLabel_Export/annotations/SK-A-23.xml
  Exported annotations for SK-A-54.jpg to /content/drive/MyDrive/RectLabel_Export/annotations/SK-A-54.xml
  Exported annotations for SK-A-60.jpg to /content/drive/MyDrive/RectLabel_Export/annotations/SK-A-60.xml
  Exported annotations for SK-A-61.jpg to /content/drive/MyDrive/RectLabel_Export/annotations/SK-A-61.xml
  Exported annotations for SK-A-62.jpg to /content/drive/MyDrive/RectLabel_Export/annotations/SK-A-62.xml
  Exported annotations for SK-A-87.jpg to /content/drive/MyDrive/RectLabel_Export/annotations/SK-A-87.xml
  Exported annotations for SK-A-110.jpg to /content/drive/MyDrive/RectLabel_Export/annotations/SK-A-110.xml
  